# Sesion 1 · Naturaleza de los datos masivos
### ¿Cuando un problema deja de caber en un solo computador?

**IFPN0025 · Big Data e Ingenieria de Datos** · Universidad Ean · Andres · `S01_P4_v1`

Este notebook cubre el **Nivel 1 (guiado)** y el **Nivel 2 (aplicado)**. El Nivel 3 y el reto de
negocio estan en `resultados/nivel3_matriz.md` y `resultados/reto_negocio.md`, sostenidos por las
mediciones que se producen aqui.

**Como se ejecuta:** *Kernel → Restart & Run All*. Si hay internet descarga las fuentes reales; si no,
activa el plan de contingencia de la seccion 2.4 y lo declara.

## 1.2 · Celda de verificacion del entorno

Si algo falla aqui, todo lo demas fallara despues.

In [1]:
import sys, platform
print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

required_packages = ["pandas", "numpy", "psutil"]
missing_packages = []
for package_name in required_packages:
    try:
        __import__(package_name); print(f"OK · {package_name}")
    except ImportError:
        missing_packages.append(package_name); print(f"FALTA · {package_name}")
print("\nInstale con:  pip install " + " ".join(missing_packages) if missing_packages else "\nEntorno listo.")

Python: 3.10.12
Sistema: Linux 6.8.0-124-generic


OK · pandas
OK · numpy
OK · psutil

Entorno listo.


In [2]:
import os, sys, math, json, time
import numpy as np, pandas as pd, psutil

def _hallar_raiz():
    """Encuentra la raiz del proyecto sin importar desde donde se lance el kernel.

    Jupyter clasico arranca el kernel en notebooks/, pero VS Code lo arranca en la
    carpeta del workspace. En vez de suponer, se busca hacia arriba la carpeta que
    contenga scripts/bd_s01.py.
    """
    candidatos = [os.getcwd()]
    archivo_vsc = globals().get("__vsc_ipynb_file__")
    if archivo_vsc:
        candidatos.append(os.path.dirname(os.path.dirname(os.path.abspath(archivo_vsc))))
    for inicio in candidatos:
        d = os.path.abspath(inicio)
        for _ in range(6):
            if os.path.isfile(os.path.join(d, "scripts", "bd_s01.py")):
                return d
            for sub in ("sesion01", "S01_entrega"):
                p = os.path.join(d, sub)
                if os.path.isfile(os.path.join(p, "scripts", "bd_s01.py")):
                    return p
            padre = os.path.dirname(d)
            if padre == d:
                break
            d = padre
    return None

RAIZ = _hallar_raiz()
if RAIZ is None:
    raise RuntimeError(
        "No encuentro scripts/bd_s01.py.\n"
        f"Directorio actual: {os.getcwd()}\n"
        "Ejecute antes:  import os; os.chdir(r'RUTA_A\\sesion01')")

sys.path.insert(0, os.path.join(RAIZ, "scripts"))
import bd_s01 as bd

for d in ["data/raw", "data/synthetic", "resultados"]:
    os.makedirs(os.path.join(RAIZ, d), exist_ok=True)
print("Raiz del proyecto :", RAIZ)
print("Directorio actual :", os.getcwd())

Raiz del proyecto : /sessions/sleepy-tender-tesla/mnt/Desktop/Universidad/IFPN0025_BigData_IngenieriaDatos/S01_naturaleza_datos_masivos/sesion01
Directorio actual : /sessions/sleepy-tender-tesla/mnt/Desktop/Universidad/IFPN0025_BigData_IngenieriaDatos/S01_naturaleza_datos_masivos/sesion01/notebooks


## 3 · Las tres mediciones que sostienen todo

Definidas en `scripts/bd_s01.py` para que el notebook y los scripts usen exactamente el mismo codigo.

- **S0** — tamano real en disco, en GB.
- **k** — factor de expansion: cuantas veces crece el archivo al cargarse en memoria.
- **M** — memoria **util**, no memoria total.

> **La advertencia que decide el resultado.** `k` se mide con `df.memory_usage(deep=True).sum()`.
> Sin `deep=True`, pandas reporta 8 bytes por celda de texto (el puntero) y no la cadena apuntada.
> Todos los `k` convergen entonces a un valor bajo y falso, y **siempre por debajo**: el error empuja
> la decision hacia *todavia no hace falta invertir*, que es el sesgo mas peligroso posible.

In [3]:
import inspect
for fn in (bd.get_file_size_gb, bd.measure_expansion_factor, bd.get_available_memory_gb):
    print(inspect.getsource(fn))

def get_file_size_gb(file_path):
    """S0 - Tamano real del archivo en disco, en gigabytes."""
    return os.path.getsize(file_path) / (1024 ** 3)

def measure_expansion_factor(file_path, **read_options):
    """k - Cuantas veces crece un archivo al cargarse en memoria.

    deep=True es obligatorio: sin el, pandas solo cuenta los punteros de las
    columnas object, no las cadenas apuntadas, y todos los k convergen a un
    valor bajo y falso.
    """
    size_disk_bytes = os.path.getsize(file_path)
    df = pd.read_csv(file_path, **read_options)
    size_memory_bytes = df.memory_usage(deep=True).sum()
    return size_memory_bytes / size_disk_bytes, df

def get_available_memory_gb():
    """M - Memoria realmente disponible, no la de la etiqueta."""
    return psutil.virtual_memory().available / (1024 ** 3)



### M · memoria util del equipo

`M` es lo que queda, no lo que dice la etiqueta.

In [4]:
m_total = bd.get_total_memory_gb()
m_disp  = bd.get_available_memory_gb()
frac_so = 1 - m_disp / m_total
print(f"RAM total        : {m_total:6.2f} GB")
print(f"RAM disponible(M): {m_disp:6.2f} GB")
print(f"Consumido por SO y aplicaciones: {frac_so:.1%}")
print(f"\nEscenario  8 GB -> M util = {bd.memoria_util(8,  frac_so):5.2f} GB")
print(f"Escenario 16 GB -> M util = {bd.memoria_util(16, frac_so):5.2f} GB")

RAM total        :   3.82 GB
RAM disponible(M):   3.30 GB
Consumido por SO y aplicaciones: 13.8%

Escenario  8 GB -> M util =  6.90 GB
Escenario 16 GB -> M util = 13.79 GB


## 2 · Las tres fuentes

Identificadores **verificados** contra la API de Socrata de `www.datos.gov.co` el 2026-07-24:

| Fuente | Identificador | Columnas | Filas totales | V esperada |
|---|---|---|---|---|
| SECOP II · Procesos de Contratacion | `p6dx-8zbt` | 59 | 8.878.158 | Volumen |
| IDEAM · Temperatura Ambiente del Aire | `sbwg-7ju4` | 12 | 21.710 registros **por dia** | Velocidad |
| GEIH · DANE | *sin API* | multiples archivos | por periodo | Variedad |

Patron de descarga: `https://www.datos.gov.co/resource/<ID>.csv?$limit=<N>`

### Nivel 1 · Pasos 1.1 y 1.2 · perfilamiento y medicion de k

El pipeline completo vive en `scripts/ejecutar_todo.py` para que sea reproducible fuera del notebook.
Esta celda **reutiliza** la corrida ya archivada; solo dispara el pipeline si aun no existe. Asi el
notebook se re-ejecuta en segundos y no vuelve a descargar 265 MB cada vez.

In [5]:
import subprocess
RUTA_JSON = os.path.join(RAIZ, "resultados", "_resultados.json")
if os.path.exists(RUTA_JSON):
    print("Corrida existente encontrada. Se reutiliza (no se vuelve a descargar).")
    print("Para forzar una medicion nueva:  python scripts/ejecutar_todo.py --redescargar")
else:
    r = subprocess.run([sys.executable, os.path.join(RAIZ, "scripts", "ejecutar_todo.py")],
                       capture_output=True, text=True, cwd=RAIZ)
    print(r.stdout[-4000:]); print(r.stderr[-1500:] if r.returncode else "")

J = json.load(open(RUTA_JSON, encoding="utf-8"))
print(f"\nEquipo   : {J['equipo']['etiqueta']} · {J['equipo']['so']} · pandas {J['equipo']['pandas']}")
print(f"Momento  : {J['equipo']['momento']}   Modo: {J['modo']}")
print(f"RAM      : {J['memoria']['total_gb']} GB totales · {J['memoria']['disponible_gb']} GB disponibles "
      f"({J['memoria']['fraccion_consumida_so']:.1%} consumido por el SO)")

Corrida existente encontrada. Se reutiliza (no se vuelve a descargar).
Para forzar una medicion nueva:  python scripts/ejecutar_todo.py --redescargar

Equipo   : Andr-s_16GB · Windows 11 · pandas 2.3.3
Momento  : 2026-07-24 18:38   Modo: real
RAM      : 15.64 GB totales · 2.08 GB disponibles (86.7% consumido por el SO)


#### Demostracion en vivo de la medicion de `k`

Para que el notebook no sea solo un lector de resultados, esta celda **vuelve a medir** `k` sobre el
archivo mas pequeno que haya en disco, y compara `deep=True` contra la version sin el argumento.

In [6]:
cands = []
for sub in ("data/raw", "data/synthetic"):
    d = os.path.join(RAIZ, sub)
    if os.path.isdir(d):
        for r_, _, fs in os.walk(d):
            cands += [os.path.join(r_, f) for f in fs
                      if f.lower().endswith(".csv") and os.path.getsize(os.path.join(r_, f)) > 100_000]

if cands:
    # el archivo mas grande es el de mas texto libre: es donde deep=True mas importa
    ruta = max(cands, key=os.path.getsize)
    N = 50_000
    df_demo = pd.read_csv(ruta, nrows=N, low_memory=False)
    con_deep = df_demo.memory_usage(deep=True).sum()
    sin_deep = df_demo.memory_usage(deep=False).sum()
    n_obj = int((df_demo.dtypes == "object").sum())
    print(f"Archivo   : {os.path.relpath(ruta, RAIZ)}   (primeras {len(df_demo):,} filas)")
    print(f"Columnas  : {df_demo.shape[1]}  ·  de tipo object (texto): {n_obj}")
    print()
    print(f"  memory_usage(deep=True)  = {con_deep/1024**2:9.2f} MB   <- lo que REALMENTE ocupa")
    print(f"  memory_usage(deep=False) = {sin_deep/1024**2:9.2f} MB   <- lo que pandas reporta sin el argumento")
    print()
    if n_obj:
        print(f"  Sin deep=True se subestima la memoria {con_deep/sin_deep:.1f} veces.")
        print(f"  Diferencia absoluta: {(con_deep-sin_deep)/1024**2:,.1f} MB que no se estaban contando.")
    else:
        print("  Este archivo no tiene columnas de texto, por eso ambas cifras coinciden.")
        print("  La diferencia solo aparece cuando hay columnas object.")
    print()
    print("  El error SIEMPRE va en la misma direccion: subestima. Y por lo tanto empuja la")
    print("  conclusion hacia 'todavia no hace falta invertir', que es el sesgo mas caro")
    print("  posible en una decision de infraestructura.")
    del df_demo
else:
    print("No hay archivos en data/ para la demostracion en vivo.")
    print("Ejecute:  python scripts/ejecutar_todo.py")

Archivo   : data/raw/secop_sample.csv   (primeras 50,000 filas)
Columnas  : 59  ·  de tipo object (texto): 43

  memory_usage(deep=True)  =    168.40 MB   <- lo que REALMENTE ocupa
  memory_usage(deep=False) =     22.51 MB   <- lo que pandas reporta sin el argumento

  Sin deep=True se subestima la memoria 7.5 veces.
  Diferencia absoluta: 145.9 MB que no se estaban contando.

  El error SIEMPRE va en la misma direccion: subestima. Y por lo tanto empuja la
  conclusion hacia 'todavia no hace falta invertir', que es el sesgo mas caro
  posible en una decision de infraestructura.


In [7]:
for m in J["mediciones"]:
    f = m["fuente"]; v = J["extras"][f]["veracidad"]; ve = J["extras"][f]["velocidad"]
    print(f"— {f}")
    print(f"    volumen  : {m['filas_fuente_completa']:,} filas -> S0 proy. {m['S0_proyectado_gb']:.3f} GB, "
          f"k={m['k']} -> {m['memoria_necesaria_gb']:.2f} GB de RAM")
    print(f"    variedad : {m['columnas_texto']}/{m['columnas']} columnas de texto ({m['proporcion_texto']:.0%})")
    print(f"    veracidad: {v['prop_nulos_media']:.1%} nulos medios, {v['columnas_sobre_50pct_nulas']} columnas >50% nulas")
    print(f"    velocidad: {ve.get('registros_por_hora_observados','n/a')} registros/hora observados\n")

— SECOP II
    volumen  : 8,878,158 filas -> S0 proy. 8.643 GB, k=3.09 -> 26.71 GB de RAM
    variedad : 44/59 columnas de texto (75%)
    veracidad: 12.8% nulos medios, 8 columnas >50% nulas
    velocidad: 2.02 registros/hora observados

— IDEAM
    volumen  : 55,469,050 filas -> S0 proy. 8.119 GB, k=3.28 -> 26.63 GB de RAM
    variedad : 7/12 columnas de texto (58%)
    veracidad: 0.0% nulos medios, 0 columnas >50% nulas
    velocidad: 833.45 registros/hora observados

— GEIH (sustituto sintetico)
    volumen  : 80,000 filas -> S0 proy. 0.009 GB, k=1.93 -> 0.02 GB de RAM
    variedad : 0/28 columnas de texto (0%)
    veracidad: 10.9% nulos medios, 0 columnas >50% nulas
    velocidad: n/a registros/hora observados



## 5 · Nivel 2 · Horizonte de saturacion

$$t_{umbral} = \frac{\ln\left(\dfrac{M}{k \cdot S_0}\right)}{\ln(1+g)}$$

**S0 es el de la fuente completa, no el de la muestra.** La muestra siempre cabe; por eso se proyecta
con `filas_totales / filas_muestra`. Y **M es la memoria util**, no la de la etiqueta.

In [8]:
print(inspect.getsource(bd.compute_threshold_periods))
esc = list(J["memoria"]["escenarios"])
filas = []
for m in J["mediciones"]:
    f = m["fuente"]
    filas.append({"fuente": f, "k": m["k"], "S0_proy_GB": m["S0_proyectado_gb"],
                  "g": J["g"][f]["g"], **{f"t · M {e}": J["umbrales"][f][e] for e in esc}})
pd.DataFrame(filas)

def compute_threshold_periods(memory_useful_gb, expansion_factor,
                              initial_size_gb, growth_rate):
    """t_umbral = ln( M / (k * S0) ) / ln(1 + g).

    Resultado negativo = el umbral YA fue superado.
    """
    if growth_rate <= 0:
        raise ValueError("La tasa de crecimiento debe ser mayor que cero.")
    ratio = memory_useful_gb / (expansion_factor * initial_size_gb)
    return math.log(ratio) / math.log(1 + growth_rate)



,fuente,k,S0_proy_GB,g,t · M 8 GB,t · M 16 GB,t · M medido (16 GB)
0,SECOP II,3.09,8.6435,0.124,-14.00,-8.07,-21.83
1,IDEAM,3.28,8.1190,0.080,-21.22,-12.22,-33.11
2,GEIH (sustituto sintetico),1.93,0.0086,0.030,194.43,217.88,163.47


### Paso 2.3 · sensibilidad a `g` y a `k`

Un `t_umbral` negativo no es un error: significa que la fuente **ya no cabe hoy**.

In [9]:
print("Sensibilidad a g (M = escenario de 16 GB):")
display(pd.DataFrame(J["sensibilidad_g"]).T)
print("\nSensibilidad a k (misma fuente, mismo g):")
display(pd.DataFrame([J["sensibilidad_k"]]).T.rename(columns={0: "t_umbral (anos)"}))

Sensibilidad a g (M = escenario de 16 GB):


,1%,2%,4%,8%,16%,32%
SECOP II,-94.8,-47.6,-24.0,-12.3,-6.4,-3.4
IDEAM,-94.5,-47.5,-24.0,-12.2,-6.3,-3.4
GEIH (sustituto sintetico),647.2,325.2,164.2,83.7,43.4,23.2



Sensibilidad a k (misma fuente, mismo g):


,t_umbral (anos)
k x0.5,-2.1
k x1.0,-8.1
k x2.0,-14.0
k x4.0,-19.9


**Lectura de las dos tablas.** `g` entra por `ln(1+g)` en el **denominador** y `k` por
`ln(k)` en el **numerador**. Por eso un error en `g` reescala el horizonte de forma suave, mientras
que un error en `k` puede **cambiarle el signo** — es decir, cambiar la decision de *tengo tiempo* a
*ya no cabe*. Y `k` es medible hoy en cinco minutos; `g` siempre es una proyeccion. El desarrollo
completo esta en `resultados/nivel2_sensibilidad.md`.

### Entregables generados

Los documentos se escriben desde las cifras medidas: no hay numeros escritos a mano en ninguno.

In [10]:
for f in sorted(os.listdir(os.path.join(RAIZ, "resultados"))):
    ruta = os.path.join(RAIZ, "resultados", f)
    if os.path.isfile(ruta):
        print(f"  {f:<38} {os.path.getsize(ruta)/1024:7.1f} KB")
print("\nPara reescribirlos:  python scripts/generar_entregables.py")

  _resultados.json                           9.7 KB
  mediciones.csv                             0.4 KB
  nivel1_paso1_3_v_dominante.md              4.2 KB
  nivel2_sensibilidad.md                    15.7 KB
  nivel3_matriz.md                           9.2 KB
  proyeccion_umbral.csv                      0.3 KB
  reto_negocio.md                            4.0 KB

Para reescribirlos:  python scripts/generar_entregables.py
